<a href="https://colab.research.google.com/github/Lakshya-Pundhir/HCL-traning-project/blob/main/banking_transaction_pattern_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Banking Transaction Pattern Analysis
### Project 10 — HCL Training / FinTech Analytics

**Author:** Lakshya  
**Dataset:** `10_bank_transactions.csv` (synthetic)  
**Specification:** Software Requirements Specification (SRS) Version 1.0  

> **Important Disclaimer:** The dataset is synthetic. The `Flagged` column is a supplied label only and is **not proof of fraud**. No output of this project may be used to claim, imply or investigate fraud by any person or institution.

In [1]:
# ============================================================
# PROJECT 10: BANKING TRANSACTION PATTERN ANALYSIS
# ============================================================

import os
import re
import pandas as pd
import numpy as np

In [2]:
# ------------------------------------------------------------
# 1. LOAD AND INSPECT DATASET (FR-01)
# ------------------------------------------------------------

file_name = '10_bank_transactions.csv'
if not os.path.exists(file_name):
    from google.colab import files
    print('Upload 10_bank_transactions.csv')
    uploaded = files.upload()
    file_name = next(iter(uploaded))

df = pd.read_csv(file_name)

print('FIRST 5 RECORDS:')
display(df.head())

print('\nDATASET INFORMATION:')
df.info()

print('\nDATASET SHAPE:', df.shape)

print('\nORIGINAL COLUMN NAMES:')
print(df.columns.tolist())

# Standardize column names
def normalize_name(name):
    return re.sub(r'[^a-z0-9]', '', str(name).lower())

aliases = {
    'Transaction_ID': ['transaction_id', 'transactionid', 'txn_id', 'id'],
    'Transaction_Amount': ['transaction_amount', 'amount', 'transaction_value', 'txn_amount'],
    'Transaction_Type': ['transaction_type', 'type', 'txn_type'],
    'Channel': ['channel', 'transaction_channel', 'payment_channel'],
    'City': ['city', 'transaction_city', 'location'],
    'Flagged': ['flagged', 'is_flagged', 'flag', 'alert_flag'],
    'Account_Type': ['account_type', 'accounttype', 'account']
}

available_columns = {normalize_name(col): col for col in df.columns}
rename_columns = {}
for standard_name, possible_names in aliases.items():
    candidates = [standard_name] + possible_names
    for name in candidates:
        key = normalize_name(name)
        if key in available_columns:
            rename_columns[available_columns[key]] = standard_name
            break

df.rename(columns=rename_columns, inplace=True)
print('\nSTANDARDIZED COLUMN NAMES:')
print(df.columns.tolist())


FIRST 5 RECORDS:


,Transaction_ID,Transaction_Date,Account_ID,Account_Type,Transaction_Type,Channel,City,Transaction_Amount,Flagged
0,TXN100000,2025-02-02,ACC1159,Salary,Debit,UPI,Ghaziabad,1427.18,0
1,TXN100001,2025-10-10,ACC1043,Business,Debit,Mobile App,Chennai,946.72,0
2,TXN100002,2025-08-27,ACC1354,Business,Credit,POS,Ghaziabad,12299.75,0
3,TXN100003,2025-06-10,ACC1296,Current,Debit,Mobile App,Kolkata,32215.50,0
4,TXN100004,2025-06-08,ACC1254,Current,Debit,UPI,Chennai,2372.80,0


Upload 10_bank_transactions.csv
Saving 10_bank_transactions.csv to 10_bank_transactions.csv
FIRST 5 RECORDS:

DATASET INFORMATION:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Transaction_ID      1000 non-null   object 
 1   Transaction_Date    1000 non-null   object 
 2   Account_ID          1000 non-null   object 
 3   Account_Type        1000 non-null   object 
 4   Transaction_Type    1000 non-null   object 
 5   Channel             1000 non-null   object 
 6   City                1000 non-null   object 
 7   Transaction_Amount  970 non-null    float64
 8   Flagged             1000 non-null   int64  
dtypes: float64(1), int64(1), object(7)
memory usage: 70.4+ KB

DATASET SHAPE: (1000, 9)

ORIGINAL COLUMN NAMES:
['Transaction_ID', 'Transaction_Date', 'Account_ID', 'Account_Type', 'Transaction_Type', 'Channel', 'City', 'Tra

In [3]:
# ------------------------------------------------------------
# 2. FIND AND HANDLE MISSING TRANSACTION AMOUNTS (FR-02)
# ------------------------------------------------------------

df['Transaction_Amount'] = pd.to_numeric(
    df['Transaction_Amount'].astype('string').str.replace(r'[₹,$,\s]', '', regex=True),
    errors='coerce'
)

missing_amounts = df['Transaction_Amount'].isnull().sum()
print('MISSING TRANSACTION AMOUNTS:', missing_amounts)

if df['Transaction_Amount'].notna().sum() == 0:
    raise ValueError('All transaction amounts are missing.')

median_amount = df['Transaction_Amount'].median()
df['Transaction_Amount'] = df['Transaction_Amount'].fillna(median_amount)

print('Missing amounts handled using median:', median_amount)


MISSING TRANSACTION AMOUNTS: 30
Missing amounts handled using median: 3581.51


In [4]:
# ------------------------------------------------------------
# 3. CHECK DUPLICATE TRANSACTION IDs (FR-03)
# ------------------------------------------------------------

duplicate_mask = df['Transaction_ID'].notna() & df['Transaction_ID'].duplicated(keep=False)
duplicate_ids = df.loc[duplicate_mask, 'Transaction_ID'].nunique()
duplicate_rows = duplicate_mask.sum()

print('DUPLICATE TRANSACTION IDs:', duplicate_ids)
print('ROWS WITH DUPLICATE IDs:', duplicate_rows)

# Keep the first record for each non-missing transaction ID
remove_duplicate_mask = df['Transaction_ID'].notna() & df['Transaction_ID'].duplicated(keep='first')
df = df.loc[~remove_duplicate_mask].copy()

print('Records after duplicate handling:', len(df))

# ------------------------------------------------------------
# PREPARE DATA FOR ANALYSIS
# ------------------------------------------------------------

def normalize_transaction_type(value):
    if pd.isna(value):
        return 'Unknown'
    value = str(value).strip().lower()
    mapping = {'credit': 'Credit', 'cr': 'Credit', 'deposit': 'Credit', 'debit': 'Debit', 'dr': 'Debit', 'withdrawal': 'Debit'}
    return mapping.get(value, value.title())

def normalize_flag(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().lower()
    if value in ['1', 'true', 'yes', 'y', 'flagged']:
        return True
    if value in ['0', 'false', 'no', 'n', 'not flagged', 'unflagged', 'normal']:
        return False
    return pd.NA

df['Transaction_Type'] = df['Transaction_Type'].apply(normalize_transaction_type)
df['Flagged'] = pd.Series([normalize_flag(v) for v in df['Flagged']], index=df.index, dtype='boolean')

for col in ['Channel', 'City', 'Account_Type']:
    df[col] = df[col].fillna('Unknown').astype('string').str.strip()

df['Amount_Abs'] = df['Transaction_Amount'].abs()


DUPLICATE TRANSACTION IDs: 5
ROWS WITH DUPLICATE IDs: 10
Records after duplicate handling: 995


In [5]:
# ------------------------------------------------------------
# 4. CALCULATE TOTAL CREDITS AND TOTAL DEBITS (FR-04)
# ------------------------------------------------------------

total_credits = df.loc[df['Transaction_Type'] == 'Credit', 'Amount_Abs'].sum()
total_debits = df.loc[df['Transaction_Type'] == 'Debit', 'Amount_Abs'].sum()

print('TOTAL CREDIT VALUE:', total_credits)
print('TOTAL DEBIT VALUE:', total_debits)


TOTAL CREDIT VALUE: 4544679.75
TOTAL DEBIT VALUE: 5589877.029999999


In [6]:
# ------------------------------------------------------------
# 5. MEAN, MEDIAN AND STANDARD DEVIATION (FR-05)
# ------------------------------------------------------------

mean_amount = df['Amount_Abs'].mean()
median_transaction = df['Amount_Abs'].median()
std_deviation = df['Amount_Abs'].std()

print('TRANSACTION STATISTICS')
print('Mean:', mean_amount)
print('Median:', median_transaction)
print('Standard Deviation:', std_deviation)


TRANSACTION STATISTICS
Mean: 10185.484201005025
Median: 3581.51
Standard Deviation: 41539.33011664516


In [7]:
# ------------------------------------------------------------
# 6. COMPARE TRANSACTION VOLUME BY CHANNEL (FR-06)
# ------------------------------------------------------------

channel_summary = df.groupby('Channel').agg(
    Transaction_Count=('Transaction_ID', 'size'),
    Total_Value=('Amount_Abs', 'sum'),
    Average_Value=('Amount_Abs', 'mean')
).reset_index()

channel_summary = channel_summary.sort_values('Transaction_Count', ascending=False)

print('TRANSACTION CHANNEL SUMMARY:')
display(channel_summary)


TRANSACTION CHANNEL SUMMARY:


,Channel,Transaction_Count,Total_Value,Average_Value
5,UPI,311,3149922.11,10128.366913
2,Mobile App,197,2573568.89,13063.801472
3,Net Banking,155,1010033.29,6516.343806
4,POS,149,1141050.90,7658.059732
0,ATM,105,750679.70,7149.330476
1,Branch,78,1509301.89,19350.024231


In [8]:
# ------------------------------------------------------------
# 7. TOTAL TRANSACTION VALUE BY CITY (FR-07)
# ------------------------------------------------------------

city_summary = df.groupby('City').agg(
    Transaction_Count=('Transaction_ID', 'size'),
    Total_Transaction_Value=('Amount_Abs', 'sum')
).reset_index()

city_summary = city_summary.sort_values('Total_Transaction_Value', ascending=False)

print('CITY TRANSACTION SUMMARY:')
display(city_summary)


CITY TRANSACTION SUMMARY:


,City,Transaction_Count,Total_Transaction_Value
6,Mumbai,135,1954548.74
7,Pune,119,1630580.38
1,Chennai,128,1400657.07
3,Ghaziabad,119,1340181.90
0,Bengaluru,125,1184160.36
4,Hyderabad,118,1163803.28
5,Kolkata,133,907001.68
2,Delhi,118,553623.37


In [9]:
# ------------------------------------------------------------
# 8. FLAGGED TRANSACTION COUNT AND PERCENTAGE (FR-08)
# ------------------------------------------------------------

valid_flag_count = int(df['Flagged'].notna().sum())
flagged_count = int(df['Flagged'].eq(True).sum())
flagged_percentage = (flagged_count / valid_flag_count) * 100 if valid_flag_count > 0 else 0

flagged_summary = pd.DataFrame({
    'Metric': ['Total Transactions', 'Valid Flag Labels', 'Flagged Transactions', 'Flagged Percentage'],
    'Value': [len(df), valid_flag_count, flagged_count, round(flagged_percentage, 2)]
})

print('FLAGGED TRANSACTION SUMMARY:')
display(flagged_summary)


FLAGGED TRANSACTION SUMMARY:


,Metric,Value
0,Total Transactions,995.00
1,Valid Flag Labels,995.00
2,Flagged Transactions,44.00
3,Flagged Percentage,4.42


In [10]:
# ------------------------------------------------------------
# 9. FLAGGED PERCENTAGE BY CHANNEL (FR-09)
# ------------------------------------------------------------

channel_flags = df.groupby('Channel').agg(
    Total_Transactions=('Transaction_ID', 'size'),
    Flagged_Count=('Flagged', lambda x: x.eq(True).sum()),
    Valid_Flag_Labels=('Flagged', 'count')
).reset_index()

channel_flags['Flagged_Percentage'] = np.where(
    channel_flags['Valid_Flag_Labels'] > 0,
    channel_flags['Flagged_Count'] / channel_flags['Valid_Flag_Labels'] * 100,
    np.nan
)

print('FLAGGED PERCENTAGE BY CHANNEL:')
display(channel_flags)


FLAGGED PERCENTAGE BY CHANNEL:


,Channel,Total_Transactions,Flagged_Count,Valid_Flag_Labels,Flagged_Percentage
0,ATM,105,4,105,3.809524
1,Branch,78,4,78,5.128205
2,Mobile App,197,9,197,4.568528
3,Net Banking,155,10,155,6.451613
4,POS,149,4,149,2.684564
5,UPI,311,13,311,4.180064


In [11]:
# ------------------------------------------------------------
# 10. FIND THE 10 LARGEST TRANSACTIONS (FR-10)
# ------------------------------------------------------------

largest_transactions = df.nlargest(10, 'Amount_Abs')

print('10 LARGEST TRANSACTIONS:')
display(largest_transactions[['Transaction_ID', 'Transaction_Type', 'Transaction_Amount', 'Channel', 'City']])


10 LARGEST TRANSACTIONS:


,Transaction_ID,Transaction_Type,Transaction_Amount,Channel,City
843,TXN100843,Credit,499612.04,UPI,Pune
776,TXN100776,Debit,493085.62,Branch,Pune
841,TXN100841,Credit,440722.19,Mobile App,Mumbai
522,TXN100522,Credit,435887.02,Branch,Chennai
197,TXN100197,Debit,360924.23,UPI,Ghaziabad
613,TXN100613,Debit,345497.42,Mobile App,Hyderabad
307,TXN100307,Debit,342371.83,Mobile App,Mumbai
108,TXN100108,Debit,342244.47,Mobile App,Bengaluru
25,TXN100025,Credit,297590.99,UPI,Ghaziabad
67,TXN100067,Credit,246139.94,UPI,Mumbai


In [12]:
# ------------------------------------------------------------
# 11. CREDIT AND DEBIT TOTALS BY ACCOUNT TYPE (FR-11)
# ------------------------------------------------------------

account_summary = df.groupby(['Account_Type', 'Transaction_Type'])['Amount_Abs'].sum().unstack(fill_value=0)

print('CREDIT AND DEBIT TOTALS BY ACCOUNT TYPE:')
display(account_summary)


CREDIT AND DEBIT TOTALS BY ACCOUNT TYPE:


Transaction_Type,Credit,Debit
Account_Type,,
Business,767124.81,899182.72
Current,1739034.29,1365861.95
Salary,994329.39,1431457.60
Savings,1044191.26,1893374.76


In [13]:
# ------------------------------------------------------------
# 12. IDENTIFY OUTLIERS USING IQR (FR-12)
# ------------------------------------------------------------

q1 = np.percentile(df['Amount_Abs'], 25)
q3 = np.percentile(df['Amount_Abs'], 75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = df[(df['Amount_Abs'] < lower_bound) | (df['Amount_Abs'] > upper_bound)]

print('IQR OUTLIER ANALYSIS')
print('Q1:', q1)
print('Q3:', q3)
print('IQR:', iqr)
print('Lower Bound:', lower_bound)
print('Upper Bound:', upper_bound)
print('Number of Outliers:', len(outliers))

display(outliers.head(10))


IQR OUTLIER ANALYSIS
Q1: 1976.9099999999999
Q3: 6252.06
IQR: 4275.150000000001
Lower Bound: -4435.8150000000005
Upper Bound: 12664.785
Number of Outliers: 91


,Transaction_ID,Transaction_Date,Account_ID,Account_Type,Transaction_Type,Channel,City,Transaction_Amount,Flagged,Amount_Abs
3,TXN100003,2025-06-10,ACC1296,Current,Debit,Mobile App,Kolkata,32215.50,False,32215.50
7,TXN100007,2025-09-12,ACC1112,Salary,Debit,POS,Hyderabad,230378.32,False,230378.32
25,TXN100025,2025-08-24,ACC1030,Current,Credit,UPI,Ghaziabad,297590.99,False,297590.99
43,TXN100043,2025-05-10,ACC1027,Business,Credit,UPI,Pune,31115.92,True,31115.92
46,TXN100046,2025-06-12,ACC1013,Current,Credit,POS,Kolkata,228933.21,False,228933.21
65,TXN100065,2025-04-29,ACC1020,Salary,Debit,ATM,Chennai,16826.55,False,16826.55
67,TXN100067,2025-05-16,ACC1020,Savings,Credit,UPI,Mumbai,246139.94,False,246139.94
95,TXN100095,2025-04-16,ACC1094,Savings,Debit,POS,Ghaziabad,27308.37,False,27308.37
96,TXN100096,2025-03-29,ACC1391,Salary,Credit,UPI,Mumbai,13184.65,True,13184.65
105,TXN100105,2025-10-15,ACC1206,Salary,Credit,UPI,Bengaluru,17163.61,False,17163.61


In [14]:
# ------------------------------------------------------------
# 13. CREATE TRANSACTION AMOUNT BANDS (FR-13)
# ------------------------------------------------------------

p25, p50, p75 = np.percentile(df['Amount_Abs'], [25, 50, 75])

df['Amount_Band'] = np.select(
    [
        df['Amount_Abs'] <= p25,
        df['Amount_Abs'] <= p50,
        df['Amount_Abs'] <= p75
    ],
    ['Low', 'Medium', 'High'],
    default='Very High'
)

amount_band_summary = df.groupby('Amount_Band', observed=False).agg(
    Transaction_Count=('Transaction_ID', 'size'),
    Total_Value=('Amount_Abs', 'sum')
).reindex(['Low', 'Medium', 'High', 'Very High']).reset_index()

print('TRANSACTION AMOUNT BANDS:')
display(amount_band_summary)


TRANSACTION AMOUNT BANDS:


,Amount_Band,Transaction_Count,Total_Value
0,Low,249,322858.03
1,Medium,264,747444.30
2,High,233,1121215.49
3,Very High,249,7943038.96


In [15]:
# ------------------------------------------------------------
# 14. NUMPY BOOLEAN INDEXING THRESHOLD FILTERING (FR-14)
# ------------------------------------------------------------

threshold = np.percentile(df['Amount_Abs'], 90)
print('SELECTED TRANSACTION THRESHOLD:', threshold)

# Isolate transactions above 90th percentile using Boolean indexing
above_threshold_mask = (df['Amount_Abs'] > threshold).to_numpy()
transactions_above_threshold = df[above_threshold_mask]

print('\nTRANSACTIONS ABOVE THE 90TH PERCENTILE')
print('Count:', len(transactions_above_threshold))
print('Total Value:', transactions_above_threshold['Amount_Abs'].sum())
display(transactions_above_threshold.head())


SELECTED TRANSACTION THRESHOLD: 11746.188000000002

TRANSACTIONS ABOVE THE 90TH PERCENTILE
Count: 100
Total Value: 6666414.68


,Transaction_ID,Transaction_Date,Account_ID,Account_Type,Transaction_Type,Channel,City,Transaction_Amount,Flagged,Amount_Abs,Amount_Band
2,TXN100002,2025-08-27,ACC1354,Business,Credit,POS,Ghaziabad,12299.75,False,12299.75,Very High
3,TXN100003,2025-06-10,ACC1296,Current,Debit,Mobile App,Kolkata,32215.50,False,32215.50,Very High
7,TXN100007,2025-09-12,ACC1112,Salary,Debit,POS,Hyderabad,230378.32,False,230378.32,Very High
25,TXN100025,2025-08-24,ACC1030,Current,Credit,UPI,Ghaziabad,297590.99,False,297590.99,Very High
38,TXN100038,2025-04-12,ACC1143,Savings,Debit,UPI,Pune,11814.99,False,11814.99,Very High


In [16]:
# ------------------------------------------------------------
# 15. FIVE CAUTIOUS OPERATIONAL OBSERVATIONS (FR-15)
# ------------------------------------------------------------

print('\n' + '=' * 60)
print('OPERATIONAL OBSERVATIONS AND RECOMMENDATIONS')
print('=' * 60)

# Observation 1: Channel
if not channel_summary.empty:
    top_channel = channel_summary.iloc[0]
    print(
        '\n1. CHANNEL PERFORMANCE:'
        f"\n{top_channel['Channel']} has the highest "
        f"transaction volume with "
        f"{int(top_channel['Transaction_Count'])} records."
        '\nRecommendation: Review channel capacity, '
        'availability and transaction processing performance.'
    )

# Observation 2: City transaction value
if not city_summary.empty:
    top_city = city_summary.iloc[0]
    print(
        '\n2. CITY TRANSACTION VALUE:'
        f"\n{top_city['City']} has the highest "
        f"recorded transaction value of "
        f"{top_city['Total_Transaction_Value']:.2f}."
        '\nRecommendation: Review regional transaction volume and ensure appropriate service capacity.'
    )

# Observation 3: Flagged records
print(
    '\n3. FLAGGED RECORDS:'
    f'\n{flagged_count} transactions are flagged out of '
    f'{valid_flag_count} records with valid labels '
    f'({flagged_percentage:.2f}%).'
    '\nRecommendation: Review the rules that generate '
    'these labels and validate records individually. '
    'A flagged label is not proof of fraud.'
)

# Observation 4: Account type
if not account_summary.empty:
    debit_by_account = account_summary['Debit'] if 'Debit' in account_summary.columns else pd.Series(dtype=float)
    if not debit_by_account.empty:
        top_debit_account = debit_by_account.idxmax()
        print(
            '\n4. ACCOUNT TYPE:'
            f'\n{top_debit_account} has the highest '
            f'recorded debit value '
            f'({debit_by_account.max():.2f}).'
            '\nRecommendation: Compare credit and debit '
            'patterns across account types and review '
            'whether the differences are expected.'
        )

# Observation 5: Outliers
print(
    '\n5. TRANSACTION OUTLIERS:'
    f'\nThe IQR method identified {len(outliers)} '
    'outlier records.'
    '\nRecommendation: Review these records with '
    'transaction context and data-quality checks. '
    'An outlier is not automatically suspicious.'
)

# ------------------------------------------------------------
# SAVE OUTPUT TABLES
# ------------------------------------------------------------

channel_summary.to_csv('channel_summary.csv', index=False)
city_summary.to_csv('city_summary.csv', index=False)
flagged_summary.to_csv('flagged_summary.csv', index=False)
channel_flags.to_csv('channel_flag_summary.csv', index=False)
account_summary.to_csv('account_summary.csv')
amount_band_summary.to_csv('amount_band_summary.csv', index=False)
outliers.to_csv('outliers.csv', index=False)
largest_transactions.to_csv('top_10_transactions.csv', index=False)

print('\nANALYSIS COMPLETED SUCCESSFULLY!')
print('Summary CSV files have been created.')


OPERATIONAL OBSERVATIONS AND RECOMMENDATIONS

1. CHANNEL PERFORMANCE:
UPI has the highest transaction volume with 311 records.
Recommendation: Review channel capacity, availability and transaction processing performance.

2. CITY TRANSACTION VALUE:
Mumbai has the highest recorded transaction value of 1954548.74.
Recommendation: Review regional transaction volume and ensure appropriate service capacity.

3. FLAGGED RECORDS:
44 transactions are flagged out of 995 records with valid labels (4.42%).
Recommendation: Review the rules that generate these labels and validate records individually. A flagged label is not proof of fraud.

4. ACCOUNT TYPE:
Savings has the highest recorded debit value (1893374.76).
Recommendation: Compare credit and debit patterns across account types and review whether the differences are expected.

5. TRANSACTION OUTLIERS:
The IQR method identified 91 outlier records.
Recommendation: Review these records with transaction context and data-quality checks. An outlie